# Formative 3: Probability Distributions, Bayesian Probability, and Gradient Descent

**Team Members:**
- **James Mukunzi** - Part 1: Probability Distributions
- **Favor** - Part 2: Bayesian Probability
- **Chinemerem (Lead)** - Part 3: Gradient Descent Manual Calculation
- **Ryan** - Part 4: Gradient Descent in Code

**Important:** Display outputs for each code cell when submitting.

---
## Implementation Guide

### Module Structure

All implementations go in `formative3_utils/` package:

**James Mukunzi:**
- `distribution.py` - Bivariate PDF, contour plots, 3D plots
- `data_loading.py` - `load_education_data()`

**Favor:**
- `bayesian.py` - Prior, likelihood, posterior calculations
- `data_loading.py` - `load_imdb_data()`

**Chinemerem (Lead) + All:**
- `manual_calculator.py` - Gradient verification
- **Handwritten PDF:** `manual_calculations/Part3_Manual_Gradient_Descent.pdf`

**Ryan:**
- `gradient_descent.py` - Manual GD, scipy wrapper
- `visualization.py` - Convergence plots, method comparison

### Workflow
1. Implement your functions in `formative3_utils/`
2. Test them independently
3. Run this notebook to see results
4. Add analysis in markdown cells

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import your custom modules
from formative3_utils.distribution import compute_bivariate_pdf, create_contour_plot, create_3d_plot
from formative3_utils.bayesian import calculate_prior, calculate_likelihood, calculate_posterior
from formative3_utils.gradient_descent import gradient_descent, scipy_optimize
from formative3_utils.data_loading import load_education_data, load_imdb_data
from formative3_utils.visualization import plot_convergence, compare_methods

---
# Part 1: Bivariate Normal Distribution
**Team Member: James Mukunzi**

**Tasks:**
1. Load Education in Africa dataset
2. Select two continuous variables
3. Compute bivariate normal PDF
4. Create contour and 3D visualizations
5. Analyze the relationship

## What Are We Doing?

We're analyzing **Education in Africa** data to understand how two education metrics relate to each other across different countries. For example:
- How does **enrollment rate** relate to **literacy rate**?
- Are countries with high enrollment also likely to have high literacy?
- Can we predict one metric from the other?

### The Bivariate Normal Distribution

The **bivariate normal distribution** models the joint probability of two continuous variables. It's like a 3D bell curve that shows:
- Where most countries cluster (the peak)
- How the two variables move together (correlation)
- The spread of data in each direction (variance)

### Mathematical Formula

The probability density function (PDF) for a bivariate normal distribution is:

$$
f(x, y) = \frac{1}{2\pi\sigma_x\sigma_y\sqrt{1-\rho^2}} \exp\left(-\frac{1}{2(1-\rho^2)}\left[\frac{(x-\mu_x)^2}{\sigma_x^2} - \frac{2\rho(x-\mu_x)(y-\mu_y)}{\sigma_x\sigma_y} + \frac{(y-\mu_y)^2}{\sigma_y^2}\right]\right)
$$

**Breaking it down:**
- $\mu_x, \mu_y$ = Mean values (average enrollment, average literacy)
- $\sigma_x, \sigma_y$ = Standard deviations (how spread out the data is)
- $\rho$ = Correlation coefficient (how strongly the variables relate: -1 to +1)
- The exponential term measures how far a point $(x, y)$ is from the mean, accounting for correlation

**In matrix form (what we'll use in code):**

$$
f(\mathbf{x}) = \frac{1}{2\pi|\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T\Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)
$$

Where:
- $\mathbf{x} = [x, y]^T$ = Our data point
- $\boldsymbol{\mu} = [\mu_x, \mu_y]^T$ = Mean vector
- $\Sigma$ = Covariance matrix (captures variances and correlation)
- $|\Sigma|$ = Determinant of covariance matrix

### Why This Matters

Understanding this distribution helps us:
1. **Identify patterns** in education data across African countries
2. **Predict** one metric from another
3. **Find outliers** (countries that don't fit the pattern)
4. **Make policy decisions** based on typical relationships

## 1.1 Load Dataset

In [ ]:
# TODO (James): Implement load_education_data() in formative3_utils/data_loading.py
education_data = load_education_data('data/education_africa.csv')
education_data.head()

### What to Look For
- **Number of countries**: More data = more reliable patterns
- **Available metrics**: Choose two that you think might be related
- **Missing values**: Some countries may not have all data

## 1.2 Select Variables and Compute Statistics

In [ ]:
# TODO (James): Choose two variables to analyze
variable1 = 'your_variable_1'  # e.g., 'primary_enrollment'
variable2 = 'your_variable_2'  # e.g., 'literacy_rate'

# Extract variables and compute statistics
X = education_data[[variable1, variable2]].dropna()
mean_vector = X.mean().values
cov_matrix = X.cov().values

print(f"Mean: {mean_vector}")
print(f"Covariance Matrix:\n{cov_matrix}")
print(f"Correlation: {X.corr().iloc[0, 1]:.3f}")

### Understanding the Statistics

**Mean Vector ($\boldsymbol{\mu}$)**:
- The "center" of our data
- Average enrollment and literacy across all countries

**Covariance Matrix ($\Sigma$)**:
$$
\Sigma = \begin{bmatrix}
\sigma_x^2 & \text{cov}(x,y) \\
\text{cov}(x,y) & \sigma_y^2
\end{bmatrix}
$$

- **Diagonal elements** ($\sigma_x^2, \sigma_y^2$): Variance of each variable (how spread out)
- **Off-diagonal elements**: Covariance (how variables move together)

**Correlation ($\rho$)**:
$$
\rho = \frac{\text{cov}(x,y)}{\sigma_x \sigma_y}
$$

- **+1**: Perfect positive relationship (both increase together)
- **0**: No linear relationship
- **-1**: Perfect negative relationship (one increases, other decreases)

### What to Expect
- **High correlation (>0.7)**: Strong relationship between variables
- **Moderate correlation (0.3-0.7)**: Some relationship
- **Low correlation (<0.3)**: Weak or no relationship

## 1.3 Compute Bivariate PDF

In [ ]:
# TODO (James): Implement compute_bivariate_pdf() in formative3_utils/distribution.py
x_range = np.linspace(X[variable1].min(), X[variable1].max(), 100)
y_range = np.linspace(X[variable2].min(), X[variable2].max(), 100)
X_grid, Y_grid = np.meshgrid(x_range, y_range)

Z = compute_bivariate_pdf(X_grid, Y_grid, mean_vector, cov_matrix)
print(f"PDF computed: {Z.shape}")

### What Does This Mean?

The PDF tells us the **probability density** at each point:
- **High density** (peak): Most countries cluster here
- **Low density** (edges): Few countries have these values
- The peak should be near the mean values

Think of it like a **heat map of likelihood** - where are countries most likely to fall?

## 1.4 Visualizations

In [ ]:
# TODO (James): Implement create_contour_plot() in formative3_utils/distribution.py
create_contour_plot(X_grid, Y_grid, Z, variable1, variable2, X)
plt.show()

### How to Read the Contour Plot

**Contour lines** show regions of equal probability:
- **Center (darkest)**: Highest probability - most countries here
- **Outer rings**: Lower probability - fewer countries
- **Shape of ellipse**:
  - **Circular**: Variables are independent (no correlation)
  - **Diagonal ellipse**: Variables are correlated
  - **Steep diagonal**: Strong positive correlation
  - **Shallow diagonal**: Strong negative correlation

**Actual data points** (dots) should cluster near the center.

**Expected Results**:
- If analyzing enrollment vs. literacy: Expect positive correlation (diagonal ellipse going up-right)
- Countries far from the center are outliers worth investigating

In [ ]:
# TODO (James): Implement create_3d_plot() in formative3_utils/distribution.py
create_3d_plot(X_grid, Y_grid, Z, variable1, variable2)
plt.show()

### How to Read the 3D Plot

This is the same distribution viewed from above:
- **Peak height**: Maximum probability density
- **Peak location**: Mean values (most typical country)
- **Spread**: How much variation exists
- **Tilt**: Direction of correlation

**What Good Output Looks Like**:
- Smooth bell-shaped surface
- Single peak (not multiple peaks)
- Symmetric spread (unless data is skewed)

**Rotate the plot** (if interactive) to see the correlation from different angles!

## 1.5 Analysis

**TODO: Add your interpretation**

Answer these questions:

1. **Relationship Strength**:
   - What is the correlation coefficient?
   - Is it positive or negative?
   - Is the relationship strong, moderate, or weak?

2. **Distribution Shape**:
   - Are the contours circular or elliptical?
   - What does this tell you about the correlation?
   - Are there any outlier countries far from the center?

3. **Real-World Interpretation**:
   - Example: "Countries with high primary enrollment (>80%) tend to also have high literacy rates (>70%), suggesting that school access directly improves literacy."
   - What does this relationship mean for education policy?
   - Can you predict one variable from the other?

4. **Outliers**:
   - Which countries (if any) don't fit the pattern?
   - Why might they be different?

### Example Interpretation Template

"We analyzed the relationship between [variable1] and [variable2] across [N] African countries. The correlation of [ρ] indicates a [strong/moderate/weak] [positive/negative] relationship. The elliptical contours show that countries with [high/low] [variable1] tend to have [high/low] [variable2]. This suggests that [real-world explanation]. Notable outliers include [countries], which may be due to [reasons]."

---
# Part 2: Bayesian Probability
**Team Member: Favor**

**Tasks:**
1. Load IMDb reviews dataset
2. Select keywords for analysis
3. Calculate prior probabilities
4. Calculate likelihoods
5. Apply Bayes' Theorem
6. Visualize and interpret results

## What Are We Doing?

We're using **Bayes' Theorem** to predict movie review sentiment from keywords. This is the foundation of spam filters, recommendation systems, and sentiment analysis!

### The Problem

**Question**: If a movie review contains the word "excellent", how likely is it to be positive?

We start with a **prior belief** (overall % of positive reviews), then **update** it based on **evidence** (the keyword appears).

### Bayes' Theorem Formula

$$
P(\text{Positive}|\text{Keyword}) = \frac{P(\text{Keyword}|\text{Positive}) \cdot P(\text{Positive})}{P(\text{Keyword})}
$$

**Breaking it down step-by-step:**

1. **Prior Probability** $P(\text{Positive})$:
   $$P(\text{Positive}) = \frac{\text{Number of positive reviews}}{\text{Total reviews}}$$
   - Our initial belief before seeing any keywords
   - Example: If 60% of reviews are positive, $P(\text{Positive}) = 0.60$

2. **Likelihood** $P(\text{Keyword}|\text{Positive})$:
   $$P(\text{Keyword}|\text{Positive}) = \frac{\text{Positive reviews with keyword}}{\text{Total positive reviews}}$$
   - How often does the keyword appear in positive reviews?
   - Example: If "excellent" appears in 30% of positive reviews, $P(\text{excellent}|\text{Positive}) = 0.30$

3. **Marginal Probability** $P(\text{Keyword})$:
   $$P(\text{Keyword}) = P(\text{Keyword}|\text{Positive}) \cdot P(\text{Positive}) + P(\text{Keyword}|\text{Negative}) \cdot P(\text{Negative})$$
   - Overall probability of seeing the keyword
   - Normalizes our result to ensure probabilities sum to 1

4. **Posterior Probability** $P(\text{Positive}|\text{Keyword})$:
   - Our **updated belief** after seeing the keyword
   - This is what we want to find!

### Why This Matters

Bayes' Theorem lets us:
- **Update beliefs** with new evidence
- **Classify** reviews as positive/negative
- **Quantify uncertainty** (not just "positive" but "85% likely positive")
- Build **spam filters**, **recommendation engines**, and **medical diagnosis systems**

### Real-World Example

- **Prior**: 60% of reviews are positive
- **Evidence**: Review contains "excellent"
- **Likelihood**: "Excellent" appears in 30% of positive reviews, but only 5% of negative reviews
- **Posterior**: After seeing "excellent", we're now 90% confident it's positive!

The keyword **shifted our belief** from 60% to 90%.

## 2.1 Load Dataset

In [ ]:
# TODO (Favor): Implement load_imdb_data() in formative3_utils/data_loading.py
imdb_data = load_imdb_data('data/imdb_reviews.csv')
print(f"Dataset shape: {imdb_data.shape}")
print(imdb_data['sentiment'].value_counts())
imdb_data.head()

### What to Look For
- **Class balance**: Are positive and negative reviews roughly equal?
- **Sample size**: More reviews = more reliable probabilities
- **Review length**: Longer reviews have more keywords to analyze

## 2.2 Select Keyword

In [ ]:
# TODO (Favor): Choose a keyword to analyze
keyword = 'excellent'  # Try: 'excellent', 'terrible', 'amazing', 'boring'

imdb_data['has_keyword'] = imdb_data['review'].str.contains(keyword, case=False, na=False)
print(f"Keyword '{keyword}' appears in {imdb_data['has_keyword'].sum()} reviews")
print(imdb_data.groupby('sentiment')['has_keyword'].agg(['sum', 'count']))

### What to Look For
- **Frequency**: Does the keyword appear enough times to be useful?
- **Imbalance**: Does it appear much more in positive or negative reviews?
- **Good keywords** show clear preference for one sentiment (e.g., "excellent" in positive, "terrible" in negative)

## 2.3 Calculate Probabilities

### Step 1: Prior Probabilities

**Before seeing any keywords**, what's the probability of each sentiment?

$$P(\text{Positive}) = \frac{\text{# positive reviews}}{\text{# total reviews}}$$

In [ ]:
# TODO (Favor): Implement calculate_prior() in formative3_utils/bayesian.py
prior_positive = calculate_prior(imdb_data, 'positive')
prior_negative = calculate_prior(imdb_data, 'negative')

print(f"P(positive) = {prior_positive:.4f}")
print(f"P(negative) = {prior_negative:.4f}")

### Step 2: Likelihoods

**Given a sentiment**, how likely is the keyword to appear?

$$P(\text{Keyword}|\text{Positive}) = \frac{\text{# positive reviews with keyword}}{\text{# positive reviews}}$$

In [ ]:
# TODO (Favor): Implement calculate_likelihood() in formative3_utils/bayesian.py
likelihood_positive = calculate_likelihood(imdb_data, keyword, 'positive')
likelihood_negative = calculate_likelihood(imdb_data, keyword, 'negative')

print(f"P('{keyword}'|positive) = {likelihood_positive:.4f}")
print(f"P('{keyword}'|negative) = {likelihood_negative:.4f}")

### Step 3: Marginal Probability

**Overall**, how likely is the keyword to appear (regardless of sentiment)?

$$P(\text{Keyword}) = P(\text{Keyword}|\text{Positive}) \cdot P(\text{Positive}) + P(\text{Keyword}|\text{Negative}) \cdot P(\text{Negative})$$

This is the **law of total probability** - we sum over all possible sentiments.

In [ ]:
# Calculate marginal probability
marginal = (likelihood_positive * prior_positive) + (likelihood_negative * prior_negative)
print(f"P('{keyword}') = {marginal:.4f}")

### Step 4: Posterior Probabilities (Bayes' Theorem!)

**Given the keyword**, what's the probability of each sentiment?

$$P(\text{Positive}|\text{Keyword}) = \frac{P(\text{Keyword}|\text{Positive}) \cdot P(\text{Positive})}{P(\text{Keyword})}$$

This is our **updated belief** after seeing the evidence!

In [ ]:
# TODO (Favor): Implement calculate_posterior() in formative3_utils/bayesian.py
posterior_positive = calculate_posterior(likelihood_positive, prior_positive, marginal)
posterior_negative = calculate_posterior(likelihood_negative, prior_negative, marginal)

print(f"P(positive|'{keyword}') = {posterior_positive:.4f}")
print(f"P(negative|'{keyword}') = {posterior_negative:.4f}")
print(f"\nBelief changed: {prior_positive:.1%} → {posterior_positive:.1%}")

### Understanding the Results

**Expected Results**:
- **Positive keywords** (excellent, amazing): Posterior > Prior for positive sentiment
- **Negative keywords** (terrible, boring): Posterior < Prior for positive sentiment
- **Neutral keywords** (movie, film): Posterior ≈ Prior (no information gain)

**What Makes a Good Keyword?**
- Large change from prior to posterior
- High likelihood ratio (appears much more in one sentiment)
- Appears frequently enough to be useful

## 2.4 Visualization

In [ ]:
# TODO (Favor): Create visualization comparing prior and posterior
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(['Positive', 'Negative'], [prior_positive, prior_negative], color=['green', 'red'], alpha=0.7)
ax1.set_title('Prior Probabilities')
ax1.set_ylim([0, 1])

ax2.bar(['Positive', 'Negative'], [posterior_positive, posterior_negative], color=['green', 'red'], alpha=0.7)
ax2.set_title(f'Posterior (after "{keyword}")')
ax2.set_ylim([0, 1])

plt.show()

### How to Read This Visualization

**Left plot (Prior)**:
- Our initial belief before seeing any keywords
- Based only on the overall distribution of sentiments
- Often close to 50-50 if dataset is balanced

**Right plot (Posterior)**:
- Our updated belief after seeing the keyword
- Shows how much the keyword shifted our confidence
- Bars should be more extreme (closer to 0 or 1)

**What Good Output Looks Like**:
- **Informative keyword**: Large difference between left and right plots
- **Positive keyword**: Right plot shows green bar much higher
- **Negative keyword**: Right plot shows red bar much higher
- **Neutral keyword**: Plots look similar (keyword doesn't help)

**The 50% threshold line** shows the decision boundary - above 50% we'd classify as positive.

## 2.5 Analysis

**TODO: Add your interpretation**

Answer these questions:

1. **Belief Update**:
   - What was the prior probability of positive sentiment?
   - What is the posterior probability after seeing the keyword?
   - How many percentage points did it change?

2. **Keyword Effectiveness**:
   - Is this keyword a good predictor of sentiment?
   - What is the likelihood ratio (how much more common in one sentiment)?
   - Would you use this keyword in a sentiment classifier?

3. **Real-World Application**:
   - Example: "The keyword 'excellent' increased our confidence in positive sentiment from 55% to 92%. This makes it a strong indicator. A spam filter could use similar logic: if an email contains 'winner', update the probability it's spam."
   - How could this be used in a recommendation system?
   - What other keywords would you test?

4. **Limitations**:
   - What if the keyword appears in sarcastic reviews?
   - Should we consider multiple keywords together?
   - How would you improve this classifier?

### Example Interpretation Template

"We analyzed the keyword '[keyword]' across [N] IMDb reviews. The prior probability of positive sentiment was [X]%, but after observing the keyword, the posterior probability became [Y]% - a change of [Z] percentage points. The likelihood ratio of [R] indicates that '[keyword]' is [R]x more common in [positive/negative] reviews. This makes it a [strong/moderate/weak] predictor. In practice, this could be used to [application]. However, limitations include [limitations]."

---
# Part 3: Gradient Descent - Manual Calculation
**Team Lead: Chinemerem | All Contribute**

**Tasks:**
1. Perform gradient descent by hand (3+ iterations)
2. Show all calculations with chain rule
3. Scan and save as PDF: `manual_calculations/Part3_Manual_Gradient_Descent.pdf`
4. Verify calculations with code

## What Are We Doing?

**Gradient descent** is the fundamental algorithm behind machine learning! It's how neural networks learn, how we fit lines to data, and how we optimize everything from spam filters to self-driving cars.

### The Problem

Imagine you're lost in foggy mountains and want to reach the valley (minimum). You can't see far, but you can feel which direction is downhill. **Gradient descent** says:
1. Feel which direction is steepest downhill (compute gradient)
2. Take a step in that direction (update parameters)
3. Repeat until you reach the bottom (convergence)

### Mathematical Formulas

**Function to Minimize**:
$$f(x, y) = x^2 + y^2$$

This is a simple bowl-shaped function with minimum at $(0, 0)$ where $f(0,0) = 0$.

**Gradient (Direction of Steepest Ascent)**:
$$\nabla f(x, y) = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix} = \begin{bmatrix} 2x \\ 2y \end{bmatrix}$$

**Computing the Gradient (Chain Rule)**:
- $\frac{\partial f}{\partial x} = \frac{\partial}{\partial x}(x^2 + y^2) = 2x + 0 = 2x$
- $\frac{\partial f}{\partial y} = \frac{\partial}{\partial y}(x^2 + y^2) = 0 + 2y = 2y$

**Update Rule (Gradient Descent)**:
$$\begin{bmatrix} x_{\text{new}} \\ y_{\text{new}} \end{bmatrix} = \begin{bmatrix} x_{\text{old}} \\ y_{\text{old}} \end{bmatrix} - \alpha \begin{bmatrix} 2x_{\text{old}} \\ 2y_{\text{old}} \end{bmatrix}$$

Or component-wise:
$$x_{\text{new}} = x_{\text{old}} - \alpha \cdot 2x_{\text{old}} = x_{\text{old}}(1 - 2\alpha)$$
$$y_{\text{new}} = y_{\text{old}} - \alpha \cdot 2y_{\text{old}} = y_{\text{old}}(1 - 2\alpha)$$

Where:
- $\alpha$ = **Learning rate** (step size, typically 0.01 to 0.1)
- We **subtract** the gradient because we want to go **downhill** (minimize)

### Why This Matters

This simple example teaches the core of machine learning:
- **Neural networks**: Adjust millions of weights using gradient descent
- **Linear regression**: Find the best-fit line by minimizing error
- **Logistic regression**: Classify data by minimizing loss
- **Deep learning**: Stack many layers and optimize with gradient descent

### Example Iteration

Starting at $(x, y) = (5, 5)$ with $\alpha = 0.1$:

1. **Compute function value**: $f(5, 5) = 5^2 + 5^2 = 50$
2. **Compute gradient**: $\nabla f(5, 5) = [2(5), 2(5)] = [10, 10]$
3. **Update**: 
   - $x_{\text{new}} = 5 - 0.1(10) = 5 - 1 = 4$
   - $y_{\text{new}} = 5 - 0.1(10) = 5 - 1 = 4$
4. **New value**: $f(4, 4) = 32$ (decreased from 50! ✓)

We moved closer to the minimum!

## 3.1 Function Definition

**Function:** $f(x, y) = x^2 + y^2$

**Gradient:** $\nabla f = \begin{bmatrix} 2x \\ 2y \end{bmatrix}$

**Update rule:** 
$$\begin{bmatrix} x_{\text{new}} \\ y_{\text{new}} \end{bmatrix} = \begin{bmatrix} x_{\text{old}} \\ y_{\text{old}} \end{bmatrix} - \alpha \nabla f$$

**Convergence criterion**: Stop when $|f_{\text{new}} - f_{\text{old}}| < \epsilon$ (change is tiny)

## 3.2 Manual Calculations

**See:** `manual_calculations/Part3_Manual_Gradient_Descent.pdf`

The PDF includes:
- Initial point and learning rate
- Step-by-step gradient calculations
- Parameter updates for each iteration
- Chain rule applications

## 3.3 Code Verification

In [ ]:
# TODO (Chinemerem): Use the SAME values as your manual calculations
def f(x, y):
    return x**2 + y**2

def gradient_f(x, y):
    return np.array([2*x, 2*y])

x0, y0 = 5.0, 5.0
learning_rate = 0.1
n_iterations = 5

print(f"Starting point: ({x0}, {y0})")
print(f"Initial value: f({x0}, {y0}) = {f(x0, y0)}")
print(f"Learning rate: {learning_rate}")

### What to Expect

- **Function value should decrease** each iteration (we're going downhill)
- **Position should move toward (0, 0)** (the minimum)
- **Gradient magnitude should decrease** (slope gets flatter near minimum)
- **Steps get smaller** as we approach the minimum

In [ ]:
# TODO (Chinemerem): Verify your manual calculations match the code
x, y = x0, y0

for i in range(n_iterations):
    grad = gradient_f(x, y)
    f_val = f(x, y)
    
    print(f"\nIteration {i}:")
    print(f"  Position: ({x:.6f}, {y:.6f})")
    print(f"  f(x,y) = {f_val:.6f}")
    print(f"  Gradient: ({grad[0]:.6f}, {grad[1]:.6f})")
    
    x = x - learning_rate * grad[0]
    y = y - learning_rate * grad[1]
    print(f"  New position: ({x:.6f}, {y:.6f})")

print(f"\nFinal: ({x:.6f}, {y:.6f}), f = {f(x, y):.6f}")

### Understanding the Output

**For each iteration, check**:
1. **Function value decreases**: If it increases, learning rate is too large!
2. **Position moves toward (0, 0)**: Both x and y should get smaller
3. **Gradient gets smaller**: As we approach minimum, slope flattens
4. **Improvement percentage decreases**: Early steps make big progress, later steps fine-tune

**Common Issues**:
- **Function increases**: Learning rate too large (try α = 0.01)
- **Slow convergence**: Learning rate too small (try α = 0.3)
- **Oscillation**: Learning rate at boundary of stability (try α = 0.1)

**Optimal learning rate for this function**: $\alpha < 1$ (otherwise we overshoot)

## 3.4 Comparison

**TODO: Compare code output with your manual calculations**

Create a comparison table:

| Iteration | Manual x | Code x | Manual y | Code y | Manual f(x,y) | Code f(x,y) | Match? |
|-----------|----------|--------|----------|--------|---------------|-------------|--------|
| 0         |          |        |          |        |               |             | ✓/✗    |
| 1         |          |        |          |        |               |             | ✓/✗    |
| 2         |          |        |          |        |               |             | ✓/✗    |
| 3         |          |        |          |        |               |             | ✓/✗    |

**Verification checklist**:
- [ ] Initial positions match
- [ ] Gradients match at each step
- [ ] Updates match (within rounding error)
- [ ] Final positions match
- [ ] Function values match

**Acceptable differences**: ±0.0001 due to rounding in manual calculations

**If they don't match**:
1. Check your gradient calculation (should be [2x, 2y])
2. Check your update formula (subtract, not add!)
3. Check your learning rate (same in both?)
4. Check arithmetic (easy to make mistakes by hand!)

---
# Part 4: Gradient Descent in Code
**Team Member: Ryan**

**Tasks:**
1. Implement gradient descent from scratch
2. Use SciPy optimization
3. Compare methods
4. Visualize convergence and paths
5. Compare with Part 3 manual calculations

## What Are We Doing?

Now we're implementing gradient descent **in code** and comparing it to **professional optimization libraries**. This shows:
- How to implement ML algorithms from scratch
- Why libraries like SciPy are faster and more robust
- How different optimization methods compare

### Two Test Functions

**1. Simple Function** (same as Part 3):
$$f(x, y) = x^2 + y^2$$
- Easy to optimize
- Minimum at $(0, 0)$
- Tests if our implementation works

**2. Rosenbrock Function** (challenging):
$$f(x, y) = (1-x)^2 + 100(y-x^2)^2$$
- Famous test function for optimization
- Minimum at $(1, 1)$ where $f(1,1) = 0$
- Has a narrow curved valley (hard to optimize!)
- Tests robustness of algorithms

**Rosenbrock Gradient**:
$$\nabla f = \begin{bmatrix} -2(1-x) - 400x(y-x^2) \\ 200(y-x^2) \end{bmatrix}$$

### MSE Cost Function (Machine Learning Context)

In linear regression, we minimize **Mean Squared Error**:
$$\text{MSE}(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i)^2$$

**Gradient of MSE**:
$$\nabla \text{MSE} = -\frac{2}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i)\mathbf{x}_i$$

**Gradient Descent Update for Linear Regression**:
$$\mathbf{w}_{\text{new}} = \mathbf{w}_{\text{old}} + \frac{2\alpha}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}_{\text{old}}^T\mathbf{x}_i)\mathbf{x}_i$$

This is how we **fit a line to data points** - adjust weights to minimize prediction error!

### Why This Matters

- **Manual implementation**: Understand what libraries do under the hood
- **SciPy comparison**: See why professionals use optimized libraries
- **Visualization**: Watch the algorithm learn in real-time
- **Method comparison**: Learn when to use different optimizers

## 4.1 Function Definitions

In [ ]:
# Simple function (same as Part 3)
def simple_function(params):
    x, y = params
    return x**2 + y**2

def simple_gradient(params):
    x, y = params
    return np.array([2*x, 2*y])

# Rosenbrock function (more complex)
def rosenbrock(params):
    x, y = params
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_gradient(params):
    x, y = params
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

## 4.2 Manual Gradient Descent

In [ ]:
# TODO (Ryan): Implement gradient_descent() in formative3_utils/gradient_descent.py
result_simple = gradient_descent(
    simple_function,
    simple_gradient,
    initial_point=np.array([5.0, 5.0]),
    learning_rate=0.1,
    max_iterations=100
)

print(f"Final position: {result_simple['final_point']}")
print(f"Final value: {result_simple['final_value']:.8f}")
print(f"Iterations: {result_simple['iterations']}")

### What Good Output Looks Like
- **Final position**: Very close to [0, 0] (e.g., [0.0001, 0.0001])
- **Final value**: Very close to 0 (e.g., 0.00000001)
- **Iterations**: Should converge in 50-100 iterations
- **Converged**: True (algorithm stopped because change was tiny)

In [ ]:
# Rosenbrock function (challenging)
result_rosenbrock = gradient_descent(
    rosenbrock,
    rosenbrock_gradient,
    initial_point=np.array([-1.0, 1.0]),
    learning_rate=0.001,
    max_iterations=1000
)

print(f"Final position: {result_rosenbrock['final_point']}")
print(f"Final value: {result_rosenbrock['final_value']:.8f}")
print(f"Iterations: {result_rosenbrock['iterations']}")

### What to Expect

**Rosenbrock is HARD**:
- May not reach exact minimum [1, 1]
- Takes many more iterations (hundreds to thousands)
- Needs smaller learning rate (0.001 vs 0.1)
- Shows why advanced optimizers (like BFGS) are needed

**Good result**: Within 0.1 of [1, 1] and value < 1

**Poor result**: Stuck far from minimum or didn't converge

## 4.3 SciPy Optimization

In [ ]:
# TODO (Ryan): Implement scipy_optimize() in formative3_utils/gradient_descent.py
scipy_result_simple = scipy_optimize(
    simple_function,
    simple_gradient,
    initial_point=np.array([5.0, 5.0]),
    method='BFGS'
)

print(f"SciPy result: {scipy_result_simple['final_point']}")
print(f"Value: {scipy_result_simple['final_value']:.8f}")

In [ ]:
scipy_result_rosenbrock = scipy_optimize(
    rosenbrock,
    rosenbrock_gradient,
    initial_point=np.array([-1.0, 1.0]),
    method='BFGS'
)

print(f"SciPy result: {scipy_result_rosenbrock['final_point']}")
print(f"Value: {scipy_result_rosenbrock['final_value']:.8f}")

## 4.4 Convergence Visualization

In [ ]:
# TODO (Ryan): Implement plot_convergence() in formative3_utils/visualization.py
plot_convergence(result_simple, title='Simple Function')
plt.show()

### How to Read Convergence Plots

**Y-axis**: Function value (what we're minimizing)

**X-axis**: Iteration number

**Good convergence looks like**:
- **Smooth decrease**: No jumps or increases
- **Fast initial drop**: Big improvements early
- **Flattening**: Slows down near minimum (diminishing returns)
- **Reaches plateau**: Stops changing (converged!)

**Problems to watch for**:
- **Oscillation**: Zig-zagging up and down (learning rate too large)
- **Slow decrease**: Barely moving (learning rate too small)
- **Sudden jumps**: Instability (learning rate way too large)
- **No plateau**: Didn't converge (need more iterations)

In [ ]:
plot_convergence(result_rosenbrock, title='Rosenbrock Function')
plt.show()

### Comparing the Two Functions

**Simple function**:
- Smooth exponential decay
- Converges quickly (< 100 iterations)
- Reaches very small values (< 1e-10)

**Rosenbrock function**:
- Slower, more erratic convergence
- Takes many iterations (> 500)
- May not reach exact minimum
- Shows why real ML problems need better optimizers!

## 4.5 Method Comparison

In [ ]:
# TODO (Ryan): Implement compare_methods() in formative3_utils/visualization.py
compare_methods(result_simple, scipy_result_simple, title='Simple Function')
plt.show()

### Understanding Method Comparison

This plot shows **optimization paths** - the route each algorithm took to the minimum.

**Manual Gradient Descent** (our implementation):
- Follows straight path toward minimum
- Takes many small steps
- Simple but effective for easy problems

**SciPy BFGS** (professional optimizer):
- Uses second-order information (curvature)
- Takes fewer, smarter steps
- Adapts step size automatically
- Much faster convergence

**What to Look For**:
- **Path shape**: Straight vs curved
- **Number of steps**: More dots = more iterations
- **Final position**: Both should reach center
- **Efficiency**: Which reaches minimum faster?

In [ ]:
compare_methods(result_rosenbrock, scipy_result_rosenbrock, title='Rosenbrock Function')
plt.show()

### Why SciPy is Better for Hard Problems

**Rosenbrock's narrow valley** makes simple gradient descent struggle:
- **Manual GD**: Zig-zags along valley, slow progress
- **BFGS**: Learns the valley shape, navigates efficiently

**Key Differences**:
- **BFGS** uses curvature information (like knowing the terrain)
- **GD** only uses gradient (like only knowing which way is downhill)
- **BFGS** adapts step size (speeds up in valleys, slows on curves)
- **GD** uses fixed step size (can't adapt to landscape)

**Lesson**: For real ML problems, use professional optimizers (Adam, BFGS, L-BFGS)!

## 4.6 Comparison with Part 3

In [ ]:
# Compare with Part 3 manual calculations
print("First 5 iterations (compare with Part 3):")
for i in range(min(6, len(result_simple['path']))):
    point = result_simple['path'][i]
    value = result_simple['values'][i]
    print(f"Iteration {i}: ({point[0]:.6f}, {point[1]:.6f}), f = {value:.6f}")

### Verification Checklist

Compare with your Part 3 manual work:

**Iteration 0** (starting point):
- [ ] Position matches your initial point
- [ ] Function value matches your calculation

**Iteration 1**:
- [ ] Position matches after first update
- [ ] Step size = learning_rate × gradient magnitude
- [ ] Function value decreased

**Iterations 2-5**:
- [ ] Positions match (within ±0.0001)
- [ ] Function values match
- [ ] Each step moves toward (0, 0)

**If they don't match**:
1. Check if you used the same learning rate (0.1)
2. Check if you used the same starting point (5, 5)
3. Review your gradient calculation
4. Check your update formula (subtract gradient!)

**Success**: Code validates your manual work! You understand gradient descent!

## 4.7 Analysis

**TODO: Add your interpretation**

Answer these questions:

1. **Convergence Speed**:
   - How many iterations did manual GD take vs SciPy?
   - Which function was easier to optimize?
   - Why is Rosenbrock harder?

2. **Learning Rate Effects**:
   - What happened with learning rate 0.1 vs 0.001?
   - Try changing the learning rate - what happens if it's too large (e.g., 0.6)?
   - What happens if it's too small (e.g., 0.001 on simple function)?

3. **Method Differences**:
   - How do the optimization paths differ?
   - Why is BFGS faster?
   - When would you use simple GD vs advanced optimizers?

4. **Verification with Part 3**:
   - Do the first 5 iterations match your manual calculations?
   - What did you learn from doing it by hand?
   - How does understanding the math help you use the code?

5. **Real-World Applications**:
   - Example: "Gradient descent is used to train neural networks with millions of parameters. Our simple example with 2 parameters shows the core idea: compute gradients, update parameters, repeat. In deep learning, we use variants like Adam (adaptive learning rates) and mini-batch GD (use subsets of data for speed)."
   - How would you apply this to linear regression?
   - What about logistic regression for classification?
   - Why do we need GPUs for deep learning?

### Example Interpretation Template

"We implemented gradient descent from scratch and compared it to SciPy's BFGS optimizer. On the simple function, manual GD converged in [N] iterations while BFGS took [M] iterations. The Rosenbrock function was much harder, requiring [K] iterations and a smaller learning rate of 0.001. The convergence plots show [observations]. Comparing with Part 3 manual calculations confirms our implementation is correct. This demonstrates how [real-world application]. Key lessons: [lessons learned]."

---
# Conclusion

## Summary

**TODO: Team summary of key learnings**

### Part 1: Bivariate Normal Distribution (James)
**What we learned**:
- How to model joint probability of two continuous variables
- The bivariate normal PDF formula and its components (mean, covariance, correlation)
- How to visualize relationships using contour and 3D plots
- Real-world application: Understanding education metrics across African countries

**Key insight**: [Your insight about the relationship you found]

### Part 2: Bayesian Probability (Favor)
**What we learned**:
- How to update beliefs with new evidence using Bayes' Theorem
- The difference between prior, likelihood, and posterior probabilities
- How to calculate marginal probability using the law of total probability
- Real-world application: Sentiment analysis and keyword-based classification

**Key insight**: [Your insight about how the keyword changed beliefs]

### Part 3: Manual Gradient Descent (Chinemerem + All)
**What we learned**:
- How to compute gradients using partial derivatives and chain rule
- The gradient descent update rule and how it moves toward minima
- How learning rate affects convergence speed and stability
- The importance of doing calculations by hand to understand the algorithm

**Key insight**: [Your insight from manual calculations]

### Part 4: Gradient Descent in Code (Ryan)
**What we learned**:
- How to implement optimization algorithms from scratch
- Why professional libraries (SciPy) are faster and more robust
- How different functions have different optimization difficulty
- How to visualize convergence and compare methods

**Key insight**: [Your insight about optimization methods]

## Connections Between Parts

**TODO: How do the four parts relate?**

All four parts are connected through **probability and optimization**:

1. **Probability Distributions** (Part 1) → **Bayesian Inference** (Part 2):
   - Both deal with probability and uncertainty
   - Bivariate normal models joint distributions; Bayes updates single distributions
   - Both are fundamental to machine learning (e.g., Naive Bayes classifier uses both!)

2. **Bayesian Inference** (Part 2) → **Optimization** (Parts 3-4):
   - Maximum likelihood estimation uses gradient descent to find best parameters
   - Bayesian optimization uses probability to guide search for optimal parameters
   - Both are about finding the "best" answer (highest probability or lowest error)

3. **Manual Calculation** (Part 3) → **Implementation** (Part 4):
   - Manual work builds intuition; code scales to real problems
   - Understanding the math helps debug code and choose hyperparameters
   - Both show the same algorithm at different scales

4. **All Parts** → **Machine Learning**:
   - **Part 1**: Model data distributions (feature relationships)
   - **Part 2**: Classify data (Naive Bayes, spam filters)
   - **Parts 3-4**: Train models (neural networks, regression)
   - Together: Complete ML pipeline from data to trained model!

## Real-World Applications

**Where you'll see these concepts**:

- **Bivariate Normal**: 
  - Financial modeling (stock correlations)
  - Anomaly detection (outlier identification)
  - Gaussian Mixture Models (clustering)

- **Bayes' Theorem**:
  - Spam filters (email classification)
  - Medical diagnosis (disease probability given symptoms)
  - Recommendation systems (user preference prediction)
  - A/B testing (which version is better?)

- **Gradient Descent**:
  - Training neural networks (deep learning)
  - Linear/logistic regression (supervised learning)
  - Computer vision (image recognition)
  - Natural language processing (ChatGPT, translation)

## Future Work

**TODO: Possible extensions**

### Extensions to Try:

1. **Part 1 Extensions**:
   - Analyze more variable pairs to find strongest correlations
   - Fit multivariate normal to 3+ variables
   - Use distribution to predict missing values
   - Identify outlier countries and investigate why

2. **Part 2 Extensions**:
   - Test multiple keywords and compare effectiveness
   - Build a simple Naive Bayes classifier
   - Calculate precision, recall, and F1 score
   - Try bigrams (two-word phrases) instead of single words

3. **Part 3-4 Extensions**:
   - Implement momentum (accelerated gradient descent)
   - Try adaptive learning rates (Adam optimizer)
   - Apply to real ML problem (linear regression on data)
   - Visualize optimization landscape in 3D
   - Compare more optimizers (SGD, RMSprop, Adam)

4. **Combined Project**:
   - Build a complete ML pipeline:
     1. Load data and model distributions (Part 1)
     2. Use Bayesian inference for feature selection (Part 2)
     3. Train a model using gradient descent (Parts 3-4)
     4. Evaluate and visualize results

## Key Takeaways

1. **Mathematics is the language of ML**: Understanding formulas helps you use tools effectively
2. **Visualization is crucial**: Plots reveal patterns that numbers hide
3. **Manual work builds intuition**: Doing it by hand once helps you use code forever
4. **Libraries are powerful**: But you need to understand what they're doing
5. **Everything connects**: Probability, statistics, and optimization form the foundation of AI

**Congratulations on completing Formative 3!** 🎉

You now understand core concepts that power modern machine learning and AI systems!